**Loading and combining results**  
Each model notebook (Prophet, SARIMA, XGBoost) saved its own results as a separate CSV during its Save and Run All. This notebook loads all of them back in and stacks them into a single table, tagging each row with which model produced it. Only the fair, forecast-ready results are included here. The lag-based XGBoost version was excluded from this comparison, since it relied on real values from within the test window that wouldn't be available at genuine forecast time, and so isn't directly comparable to the other three. The combined table is sorted by RMSE within each target, so the best-performing model of each target appears first.

In [1]:
import pandas as pd

base = '/kaggle/input/datasets/sanyazuberi'

results_prophet = pd.read_csv(f'{base}/ev-charging-results-prophet/results_prophet.csv')
results_sarima = pd.read_csv(f'{base}/ev-charging-results-sarima/results_sarima.csv')
results_xgb_cal = pd.read_csv(f'{base}/ev-charging-results-xgboost/results_xgboost_calendar.csv')

results_all = pd.concat([
    results_prophet.assign(model='Prophet', conditions='Blind multi-step'),
    results_sarima.assign(model='SARIMA', conditions='Blind multi-step'),
    results_xgb_cal.assign(model='XGBoost (calendar-only)', conditions='Blind multi-step')
], ignore_index=True)

results_all = results_all[['target', 'model', 'conditions', 'RMSE', 'MAPE']]
results_all = results_all.sort_values(['target', 'RMSE']).reset_index(drop=True)
results_all

,target,model,conditions,RMSE,MAPE
0,session_count,XGBoost (calendar-only),Blind multi-step,4.983864,0.349952
1,session_count,Prophet,Blind multi-step,5.314063,0.413010
2,session_count,SARIMA,Blind multi-step,5.461029,0.342386
3,total_energy_kwh,XGBoost (calendar-only),Blind multi-step,233.893227,0.369924
4,total_energy_kwh,Prophet,Blind multi-step,262.138628,0.449249
5,total_energy_kwh,SARIMA,Blind multi-step,264.722525,0.378930


XGBoost (calendar-only) has the lowest RMSE for both targets, outperforming Prophet by roughly 6% and SARIMA by roughly 9% on session count RMSE, and outperforming both by 11-13% on energy RMSE. On MAPE, XGBoost is lowest for energy, but SARIMA is marginally ahead on session count (0.342 vs. 0.350). This gap is small enough not to change the overall conclusion, especially since XGBoost still wins on RMSE for that same target.  
All three models were evaluated under identical, fair conditions. Each forecast the full 90-day test period without access to any real values from within it.